In [25]:
# Import libraries
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

In [26]:
# Set file paths
dataset = 'model/keypoint_classifier/keypoint.csv'
model_save_path = 'model/keypoint_classifier/keypoint_classifier.keras'

In [27]:
# Set class count
NUM_CLASSES = 10

In [28]:
# Load preprocessed features
# The CSV already stores normalized 42-value landmark vectors, so we load them directly.
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

In [29]:
# Load labels
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [30]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

In [31]:
# Build model
# Input expects the preprocessed 42-value feature vector from the CSV.
model = tf.keras.models.Sequential([
    tf.keras.layers.Input((21 * 2, )),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

In [32]:
# Show summary
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout_2 (Dropout)             │ (None, 42)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 20)             │           860 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           110 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,180 (4.61 KB)

 Trainable params: 1,180 (4.61 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# Set callbacks
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Early stopping callback
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [34]:
# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
# Train model
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
 1/42 ━━━━━━━━━━━━━━━━━━━━ 16s 391ms/step - accuracy: 0.1875 - loss: 2.4152
Epoch 1: saving model to model/keypoint_classifier/keypoint_classifier.keras
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1681 - loss: 2.3455 - val_accuracy: 0.2844 - val_loss: 2.1028
Epoch 2/1000
 1/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2266 - loss: 2.0990
Epoch 2: saving model to model/keypoint_classifier/keypoint_classifier.keras
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2278 - loss: 2.0979 - val_accuracy: 0.4218 - val_loss: 1.8860
Epoch 3/1000
 1/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2734 - loss: 1.9794
Epoch 3: saving model to model/keypoint_classifier/keypoint_classifier.keras
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2932 - loss: 1.9185 - val_accuracy: 0.4924 - val_loss: 1.6925
Epoch 4/1000
 1/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3828 - loss: 1.7830
Epoch 4: saving model to model/keypoint_classifier/keypoint_classifier.keras


In [36]:
# Evaluate model
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step - accuracy: 0.9551 - loss: 0.2649


In [37]:
# Reload model
# Load saved model
model = tf.keras.models.load_model(model_save_path)

In [38]:
# Run inference
# Inference test
predict_result = model.predict(np.array([X_test[0]]))
print(np.squeeze(predict_result))
print(np.argmax(np.squeeze(predict_result)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
[5.0639188e-01 4.0417171e-01 7.7683933e-02 6.0482882e-03 3.2945185e-03
 1.4649322e-03 1.2787245e-04 6.4346896e-05 6.3402922e-06 7.4610818e-04]
0


In [40]:
# Save model
# Save inference model
model.save(model_save_path, include_optimizer=False)

In [41]:
# Quantize model
tflite_save_path = 'model/keypoint_classifier/keypoint_classifier.tflite'

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

open(tflite_save_path, 'wb').write(tflite_quantized_model)

INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z/assets


INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z/assets


Saved artifact at '/var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  14411644240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14350477264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14419002640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14411844240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14417787360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14351849952: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1771757941.584776  223884 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1771757941.584788  223884 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-02-22 15:59:01.584889: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z
2026-02-22 15:59:01.585101: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-02-22 15:59:01.585105: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z
2026-02-22 15:59:01.587321: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-02-22 15:59:01.599879: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmptlhtfx0z
2026-02-22 15:59:01.603733: I tensorflow/cc/saved_model/loader.cc:

6836

In [42]:
# Load interpreter
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
interpreter.allocate_tensors()

/Users/ahmad/Library/Python/3.9/lib/python/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [43]:
# Get I/O tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [44]:
# Set input sample
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [45]:
%%time
# Run inference
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])

CPU times: user 232 µs, sys: 62 µs, total: 294 µs
Wall time: 265 µs


In [46]:
# Print predictions
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))

[5.0639182e-01 4.0417185e-01 7.7683963e-02 6.0482877e-03 3.2945180e-03
 1.4649314e-03 1.2787255e-04 6.4346888e-05 6.3402922e-06 7.4610813e-04]
0
